In [1]:
import re
from collections import Counter

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [2]:
# ----------------------------
# 1. Basic preprocessing
# ----------------------------
def preprocess_text(text: str) -> list[str]:
    """
    Basic preprocessing for IMDB reviews:
    - lowercase
    - remove HTML tags like <br />
    - keep words + basic punctuation as separate tokens
    """
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)          # remove HTML tags
    text = re.sub(r"[^a-z0-9\s.,!?;:'\"()-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # tokenize: words or punctuation
    tokens = re.findall(r"\w+|[.,!?;:'\"()-]", text)
    return tokens

In [3]:
# ----------------------------
# 2. Vocabulary
# ----------------------------
class Vocab:
    def __init__(self, min_freq: int = 2, max_size: int | None = 20000):
        self.min_freq = min_freq
        self.max_size = max_size

        self.pad_token = "<pad>"
        self.unk_token = "<unk>"

        self.stoi = {self.pad_token: 0, self.unk_token: 1}
        self.itos = [self.pad_token, self.unk_token]

    def build(self, tokenized_texts: list[list[str]]) -> None:
        counter = Counter()
        for tokens in tokenized_texts:
            counter.update(tokens)

        items = [(tok, freq) for tok, freq in counter.items() if freq >= self.min_freq]
        items.sort(key=lambda x: (-x[1], x[0]))

        if self.max_size is not None:
            max_new_tokens = self.max_size - len(self.itos)
            items = items[:max_new_tokens]

        for token, _ in items:
            self.stoi[token] = len(self.itos)
            self.itos.append(token)

    def encode(self, tokens: list[str]) -> list[int]:
        unk_idx = self.stoi[self.unk_token]
        return [self.stoi.get(tok, unk_idx) for tok in tokens]

    def decode(self, ids: list[int]) -> list[str]:
        return [self.itos[i] if 0 <= i < len(self.itos) else self.unk_token for i in ids]

    @property
    def pad_idx(self) -> int:
        return self.stoi[self.pad_token]

    @property
    def unk_idx(self) -> int:
        return self.stoi[self.unk_token]

    def __len__(self) -> int:
        return len(self.itos)


In [4]:
# ----------------------------
# 3. Dataset
# ----------------------------
class IMDBDataset(Dataset):
    def __init__(
        self,
        texts: list[str],
        labels: list[int],
        vocab: Vocab,
        max_len: int = 200
    ):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int):
        text = self.texts[idx]
        label = self.labels[idx]

        tokens = preprocess_text(text)
        input_ids = self.vocab.encode(tokens)

        # truncate
        input_ids = input_ids[:self.max_len]

        # attention mask: 1 for real tokens, 0 for padding
        attention_mask = [1] * len(input_ids)

        # pad
        pad_length = self.max_len - len(input_ids)
        if pad_length > 0:
            input_ids = input_ids + [self.vocab.pad_idx] * pad_length
            attention_mask = attention_mask + [0] * pad_length

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "label": torch.tensor(label, dtype=torch.long),
        }

In [5]:
# ----------------------------
# 4. Load dataframe and split
# ----------------------------
def load_imdb_dataframe(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # keep only needed columns
    df = df[["review", "sentiment"]].dropna().copy()

    # map labels
    label_map = {"negative": 0, "positive": 1}
    df["label"] = df["sentiment"].map(label_map)

    # drop rows with unexpected labels
    df = df.dropna(subset=["label"]).copy()
    df["label"] = df["label"].astype(int)

    return df


def build_splits(
    df: pd.DataFrame,
    test_size: float = 0.2,
    val_size: float = 0.1,
    random_state: int = 42
):
    """
    Splits:
    - test_size from full data
    - val_size from remaining training portion
    Example:
      test = 20%
      val = 10% of remaining 80% = 8% of full data
    """
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df["label"],
        random_state=random_state
    )

    relative_val_size = val_size / (1 - test_size)

    train_df, val_df = train_test_split(
        train_df,
        test_size=relative_val_size,
        stratify=train_df["label"],
        random_state=random_state
    )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

In [6]:
# ----------------------------
# 5. Build dataloaders
# ----------------------------
def create_imdb_dataloaders(
    csv_path: str,
    batch_size: int = 32,
    max_len: int = 200,
    min_freq: int = 2,
    max_vocab_size: int = 20000,
    random_state: int = 42
):
    # load + split
    df = load_imdb_dataframe(csv_path)
    train_df, val_df, test_df = build_splits(df, random_state=random_state)

    # build vocab on TRAIN ONLY
    train_tokens = [preprocess_text(text) for text in train_df["review"].tolist()]
    vocab = Vocab(min_freq=min_freq, max_size=max_vocab_size)
    vocab.build(train_tokens)

    # datasets
    train_dataset = IMDBDataset(
        texts=train_df["review"].tolist(),
        labels=train_df["label"].tolist(),
        vocab=vocab,
        max_len=max_len
    )
    val_dataset = IMDBDataset(
        texts=val_df["review"].tolist(),
        labels=val_df["label"].tolist(),
        vocab=vocab,
        max_len=max_len
    )
    test_dataset = IMDBDataset(
        texts=test_df["review"].tolist(),
        labels=test_df["label"].tolist(),
        vocab=vocab,
        max_len=max_len
    )

    # dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, vocab

In [7]:
if __name__ == "__main__":
    csv_path = "data/IMDB.csv"   # change if needed

    train_loader, val_loader, test_loader, vocab = create_imdb_dataloaders(
        csv_path=csv_path,
        batch_size=32,
        max_len=200,
        min_freq=2,
        max_vocab_size=20000,
        random_state=42
    )

    print(f"Vocab size: {len(vocab)}")
    print(f"Train batches: {len(train_loader)}")
    print(f"Val batches: {len(val_loader)}")
    print(f"Test batches: {len(test_loader)}")

    batch = next(iter(train_loader))
    print("input_ids shape:", batch["input_ids"].shape)         # [batch, max_len]
    print("attention_mask shape:", batch["attention_mask"].shape)
    print("label shape:", batch["label"].shape)

    # inspect one example
    sample_ids = batch["input_ids"][0].tolist()
    sample_tokens = vocab.decode(sample_ids[:30])
    print("First 30 decoded tokens:", sample_tokens)
    print("Label:", batch["label"][0].item())

Vocab size: 20000
Train batches: 1094
Val batches: 157
Test batches: 313
input_ids shape: torch.Size([32, 200])
attention_mask shape: torch.Size([32, 200])
label shape: torch.Size([32])
First 30 decoded tokens: ['i', 'was', 'so', 'offended', 'by', 'this', 'film', 'that', 'i', 'had', 'to', 'write', 'something', 'about', 'it', ',', 'so', 'please', 'humour', 'me', '.', 'its', 'only', 'redeeming', 'virtue', ',', 'outside', 'of', 'some', 'good']
Label: 0


## LSTM

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim


# ----------------------------
# 1. LSTM Baseline Model
# ----------------------------
class LSTMBaseline(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        num_classes: int,
        pad_idx: int,
        num_layers: int = 1,
        dropout: float = 0.3,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids):
        # input_ids: [batch, seq_len]
        embedded = self.embedding(input_ids)              # [batch, seq_len, embed_dim]
        output, (hidden, cell) = self.lstm(embedded)

        # hidden: [num_layers, batch, hidden_dim]
        final_hidden = hidden[-1]                         # [batch, hidden_dim]
        final_hidden = self.dropout(final_hidden)

        logits = self.fc(final_hidden)                    # [batch, num_classes]
        return logits


# ----------------------------
# 2. Train / Eval Helpers
# ----------------------------
def compute_accuracy(logits, labels):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    return correct, total


def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        correct, total = compute_accuracy(logits, labels)
        total_correct += correct
        total_examples += total

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        correct, total = compute_accuracy(logits, labels)
        total_correct += correct
        total_examples += total

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc


# ----------------------------
# 3. Full Training Script
# ----------------------------
def run_training(
    train_loader,
    val_loader,
    test_loader,
    vocab,
    num_epochs: int = 5,
    embed_dim: int = 128,
    hidden_dim: int = 128,
    lr: float = 1e-3,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    model = LSTMBaseline(
        vocab_size=len(vocab),
        embed_dim=embed_dim,
        hidden_dim=hidden_dim,
        num_classes=2,
        pad_idx=vocab.pad_idx,
        num_layers=1,
        dropout=0.3,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val_acc = 0.0
    best_model_state = None

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print(f"\nBest Val Acc: {best_val_acc:.4f}")
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    return model


# ----------------------------
# 4. Example Usage
# ----------------------------
if __name__ == "__main__":
    # make sure create_imdb_dataloaders is already defined/imported
    train_loader, val_loader, test_loader, vocab = create_imdb_dataloaders(
        csv_path="data/IMDB.csv",
        batch_size=32,
        max_len=200,
        min_freq=2,
        max_vocab_size=20000,
        random_state=42
    )

    model = run_training(
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        vocab=vocab,
        num_epochs=5,
        embed_dim=128,
        hidden_dim=128,
        lr=1e-3
    )

Using device: cpu
Epoch 01 | Train Loss: 0.6923 | Train Acc: 0.5199 | Val Loss: 0.6892 | Val Acc: 0.5322
Epoch 02 | Train Loss: 0.6629 | Train Acc: 0.6070 | Val Loss: 0.6618 | Val Acc: 0.5910
Epoch 03 | Train Loss: 0.6121 | Train Acc: 0.6689 | Val Loss: 0.7023 | Val Acc: 0.5756
Epoch 04 | Train Loss: 0.4671 | Train Acc: 0.7859 | Val Loss: 0.3721 | Val Acc: 0.8362
Epoch 05 | Train Loss: 0.2901 | Train Acc: 0.8841 | Val Loss: 0.3212 | Val Acc: 0.8634

Best Val Acc: 0.8634
Test Loss: 0.3328 | Test Acc: 0.8609


## BILSTM

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim


# ----------------------------
# 1. BiLSTM Model
# ----------------------------
class BiLSTMBaseline(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        num_classes: int,
        pad_idx: int,
        num_layers: int = 1,
        dropout: float = 0.3,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)          # [batch, seq_len, embed_dim]
        output, (hidden, cell) = self.lstm(embedded)

        forward_hidden = hidden[-2]                   # [batch, hidden_dim]
        backward_hidden = hidden[-1]                  # [batch, hidden_dim]

        final_hidden = torch.cat((forward_hidden, backward_hidden), dim=1)
        final_hidden = self.dropout(final_hidden)

        logits = self.fc(final_hidden)                # [batch, num_classes]
        return logits


# ----------------------------
# 2. Train / Eval Helpers
# ----------------------------
def compute_accuracy(logits, labels):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    return correct, total


def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        correct, total = compute_accuracy(logits, labels)
        total_correct += correct
        total_examples += total

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        correct, total = compute_accuracy(logits, labels)
        total_correct += correct
        total_examples += total

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc


# ----------------------------
# 3. Full Training Script
# ----------------------------
def run_bilstm_training(
    train_loader,
    val_loader,
    test_loader,
    vocab,
    num_epochs: int = 5,
    embed_dim: int = 128,
    hidden_dim: int = 128,
    lr: float = 1e-3,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    model = BiLSTMBaseline(
        vocab_size=len(vocab),
        embed_dim=embed_dim,
        hidden_dim=hidden_dim,
        num_classes=2,
        pad_idx=vocab.pad_idx,
        num_layers=1,
        dropout=0.3,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val_acc = 0.0
    best_model_state = None

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print(f"\nBest Val Acc: {best_val_acc:.4f}")
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    return model

In [12]:
if __name__ == "__main__":
    # make sure create_imdb_dataloaders is already defined/imported
    train_loader, val_loader, test_loader, vocab = create_imdb_dataloaders(
        csv_path="data/IMDB.csv",
        batch_size=32,
        max_len=200,
        min_freq=2,
        max_vocab_size=20000,
        random_state=42
    )

bilstm_model = run_bilstm_training(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    vocab=vocab,
    num_epochs=5,
    embed_dim=128,
    hidden_dim=128,
    lr=1e-3
)

Using device: cpu
Epoch 01 | Train Loss: 0.6340 | Train Acc: 0.6376 | Val Loss: 0.5893 | Val Acc: 0.6902


## LSTM + Attention

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


# ----------------------------
# 1. Attention Layer
# ----------------------------
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs, attention_mask=None):
        """
        lstm_outputs: [batch, seq_len, hidden_dim]
        attention_mask: [batch, seq_len], 1 for real tokens, 0 for padding
        """
        # scores: [batch, seq_len, 1]
        scores = self.attn(lstm_outputs)

        # squeeze -> [batch, seq_len]
        scores = scores.squeeze(-1)

        # mask padding positions before softmax
        if attention_mask is not None:
            scores = scores.masked_fill(attention_mask == 0, float("-inf"))

        # weights: [batch, seq_len]
        attn_weights = torch.softmax(scores, dim=1)

        # context: weighted sum of hidden states
        # attn_weights.unsqueeze(1): [batch, 1, seq_len]
        # lstm_outputs: [batch, seq_len, hidden_dim]
        # result: [batch, 1, hidden_dim] -> squeeze -> [batch, hidden_dim]
        context = torch.bmm(attn_weights.unsqueeze(1), lstm_outputs).squeeze(1)

        return context, attn_weights


# ----------------------------
# 2. LSTM + Attention Model
# ----------------------------
class LSTMAttentionModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        num_classes: int,
        pad_idx: int,
        num_layers: int = 1,
        dropout: float = 0.3,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False
        )

        self.attention = AttentionLayer(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids, attention_mask=None, return_attention=False):
        # input_ids: [batch, seq_len]
        embedded = self.embedding(input_ids)                    # [batch, seq_len, embed_dim]
        outputs, (hidden, cell) = self.lstm(embedded)          # outputs: [batch, seq_len, hidden_dim]

        context, attn_weights = self.attention(outputs, attention_mask)
        context = self.dropout(context)

        logits = self.fc(context)

        if return_attention:
            return logits, attn_weights
        return logits

In [ ]:
def compute_accuracy(logits, labels):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    return correct, total


def train_one_epoch_attention(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        correct, total = compute_accuracy(logits, labels)
        total_correct += correct
        total_examples += total

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc


@torch.no_grad()
def evaluate_attention(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        correct, total = compute_accuracy(logits, labels)
        total_correct += correct
        total_examples += total

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc

def run_lstm_attention_training(
    train_loader,
    val_loader,
    test_loader,
    vocab,
    num_epochs: int = 5,
    embed_dim: int = 128,
    hidden_dim: int = 128,
    lr: float = 1e-3,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    model = LSTMAttentionModel(
        vocab_size=len(vocab),
        embed_dim=embed_dim,
        hidden_dim=hidden_dim,
        num_classes=2,
        pad_idx=vocab.pad_idx,
        num_layers=1,
        dropout=0.3,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val_acc = 0.0
    best_model_state = None

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch_attention(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_acc = evaluate_attention(
            model, val_loader, criterion, device
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    test_loss, test_acc = evaluate_attention(model, test_loader, criterion, device)
    print(f"\nBest Val Acc: {best_val_acc:.4f}")
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    return model

In [ ]:
lstm_attn_model = run_lstm_attention_training(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    vocab=vocab,
    num_epochs=5,
    embed_dim=128,
    hidden_dim=128,
    lr=1e-3
)

## BiLSTM + Attention

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


# ----------------------------
# 1. Attention Layer
# ----------------------------
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        # 因为 BiLSTM 输出维度是 hidden_dim * 2
        self.attn = nn.Linear(hidden_dim * 2, 1)

    def forward(self, lstm_outputs, attention_mask=None):
        """
        lstm_outputs: [batch, seq_len, hidden_dim * 2]
        attention_mask: [batch, seq_len], 1 for real tokens, 0 for padding
        """
        # scores: [batch, seq_len, 1]
        scores = self.attn(lstm_outputs)

        # [batch, seq_len]
        scores = scores.squeeze(-1)

        # mask padding before softmax
        if attention_mask is not None:
            scores = scores.masked_fill(attention_mask == 0, float("-inf"))

        # [batch, seq_len]
        attn_weights = torch.softmax(scores, dim=1)

        # weighted sum
        # [batch, 1, seq_len] x [batch, seq_len, hidden_dim*2]
        # -> [batch, 1, hidden_dim*2] -> [batch, hidden_dim*2]
        context = torch.bmm(attn_weights.unsqueeze(1), lstm_outputs).squeeze(1)

        return context, attn_weights


# ----------------------------
# 2. BiLSTM + Attention Model
# ----------------------------
class BiLSTMAttentionModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        num_classes: int,
        pad_idx: int,
        num_layers: int = 1,
        dropout: float = 0.3,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True
        )

        self.attention = AttentionLayer(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, attention_mask=None, return_attention=False):
        # input_ids: [batch, seq_len]
        embedded = self.embedding(input_ids)                   # [batch, seq_len, embed_dim]
        outputs, (hidden, cell) = self.lstm(embedded)         # outputs: [batch, seq_len, hidden_dim*2]

        context, attn_weights = self.attention(outputs, attention_mask)
        context = self.dropout(context)

        logits = self.fc(context)                             # [batch, num_classes]

        if return_attention:
            return logits, attn_weights
        return logits

In [ ]:
def compute_accuracy(logits, labels):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    return correct, total


def train_one_epoch_attention(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        correct, total = compute_accuracy(logits, labels)
        total_correct += correct
        total_examples += total

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc


@torch.no_grad()
def evaluate_attention(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        correct, total = compute_accuracy(logits, labels)
        total_correct += correct
        total_examples += total

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc

def run_bilstm_attention_training(
    train_loader,
    val_loader,
    test_loader,
    vocab,
    num_epochs: int = 5,
    embed_dim: int = 128,
    hidden_dim: int = 128,
    lr: float = 1e-3,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    model = BiLSTMAttentionModel(
        vocab_size=len(vocab),
        embed_dim=embed_dim,
        hidden_dim=hidden_dim,
        num_classes=2,
        pad_idx=vocab.pad_idx,
        num_layers=1,
        dropout=0.3,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val_acc = 0.0
    best_model_state = None

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch_attention(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_acc = evaluate_attention(
            model, val_loader, criterion, device
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    test_loss, test_acc = evaluate_attention(model, test_loader, criterion, device)
    print(f"\nBest Val Acc: {best_val_acc:.4f}")
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    return model

In [ ]:
bilstm_attn_model = run_bilstm_attention_training(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    vocab=vocab,
    num_epochs=5,
    embed_dim=128,
    hidden_dim=128,
    lr=1e-3
)